<a href="https://colab.research.google.com/github/Saliyah-53/Saliyah-53/blob/main/SaliAI_Improved_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SaliAI — StanceEval2026 Improved Model**

Team **SaliAI** improved pipeline. Builds on the official baseline with 4 upgrades:

| # | Upgrade | Why |
|---|---------|-----|
| 1 | **MARBERTv2** encoder (1B Arabic tweets) | Stronger social-media Arabic representations |
| 2 | **Class-weighted loss** | Fixes Against/None imbalance |
| 3 | **Confidence-weighted loss** | Down-weights examples annotators disagreed on (`stance:confidence`) |
| 4 | **Multi-task heads**: sentiment + sarcasm + stance-explicitness | Uses the auxiliary labels the baseline ignores |

Workflow is identical to the baseline: upload `train.csv` + `dev.csv` into a `data/` folder, enable GPU, Run all.


In [1]:
# ============================================================
# 1) Install Required Packages
# ============================================================
!pip install -q transformers scikit-learn pandas tqdm sentencepiece


In [2]:
# ============================================================
# 2) Imports
# ============================================================
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm.auto import tqdm


In [4]:
# ============================================================
# 3) Configuration
# ============================================================
BASE_DIR = "."
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/train.csv"
DEV_PATH = f"{DATA_DIR}/dev.csv"
TEST_SEEN_PATH = f"{DATA_DIR}/test_seen.csv"
TEST_UNSEEN_PATH = f"{DATA_DIR}/test_unseen.csv"

OUTPUT_DIR = f"{BASE_DIR}/saliai_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Columns
TEXT_COL = "text"
TARGET_COL = "target"
LABEL_COL = "stance"
ID_COL = "id"
SENT_COL = "sentiment"
SARC_COL = "sarcasm"
CONF_COL = "stance:confidence"
FAVOR_REASON_COL = "favor_reason"
AGAINST_REASON_COL = "against_reason"

# Model
MODEL_DISPLAY_NAME = "SaliAI-MARBERTv2-MTL"
MODEL_HF_NAME = "UBC-NLP/MARBERTv2"

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 2e-5
WARMUP_RATIO = 0.1
SEED = 42

# Labels
LABEL2ID = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
SENT2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
SARC2ID = {"No": 0, "Yes": 1}
IGNORE = -100

# Auxiliary loss weights (stance loss weight = 1.0)
AUX_W_SENT = 0.3
AUX_W_SARC = 0.2
AUX_W_EXPL = 0.2

# Confidence weighting: sample_weight = 0.5 + 0.5 * confidence
CONF_FLOOR = 0.5

required_files = [TRAIN_PATH, DEV_PATH]
for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Put train.csv and dev.csv inside the data/ folder.")

print("All required dataset files found.")
print("Model:", MODEL_DISPLAY_NAME)


All required dataset files found.
Model: SaliAI-MARBERTv2-MTL


In [5]:
# ============================================================
# 4) Reproducibility
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [6]:
# ============================================================
# 5) Arabic Text Preprocessing (same as baseline)
# ============================================================
ARABIC_DIACRITICS = re.compile(r"\u0651|\u064e|\u064b|\u064f|\u064c|\u0650|\u064d|\u0652|\u0640")
NON_ARABIC = re.compile(r"[^\u0600-\u06FF0-9\s]+")
MULTI_SPACE = re.compile(r"\s+")
REPEATED_CHAR = re.compile(r"(.)\1{2,}")

def preprocess_text(text):
    text = str(text)
    text = re.sub(ARABIC_DIACRITICS, "", text)
    text = re.sub(NON_ARABIC, " ", text)
    text = re.sub(REPEATED_CHAR, r"\1\1", text)
    text = re.sub(MULTI_SPACE, " ", text).strip()
    return text


In [7]:
# ============================================================
# 6) Load Data + Build Auxiliary Labels
# ============================================================
def add_aux_labels(df):
    # Sentiment
    if SENT_COL in df.columns:
        df["sent_label"] = df[SENT_COL].map(lambda v: SENT2ID.get(str(v).strip(), IGNORE))
    else:
        df["sent_label"] = IGNORE

    # Sarcasm
    if SARC_COL in df.columns:
        df["sarc_label"] = df[SARC_COL].map(lambda v: SARC2ID.get(str(v).strip(), IGNORE))
    else:
        df["sarc_label"] = IGNORE

    # Stance explicitness (from reason columns: *_Explicit / *_Implicit)
    def expl_lab(row):
        for col in (FAVOR_REASON_COL, AGAINST_REASON_COL):
            v = str(row.get(col, "")).strip()
            if v.endswith("Explicit"):
                return 0
            if v.endswith("Implicit"):
                return 1
        return IGNORE

    if FAVOR_REASON_COL in df.columns or AGAINST_REASON_COL in df.columns:
        df["expl_label"] = df.apply(expl_lab, axis=1)
    else:
        df["expl_label"] = IGNORE

    # Annotator-confidence sample weight
    def conf_weight(v):
        try:
            c = float(v)
        except (TypeError, ValueError):
            c = 1.0
        return CONF_FLOOR + (1.0 - CONF_FLOOR) * c

    if CONF_COL in df.columns:
        df["sample_weight"] = df[CONF_COL].map(conf_weight)
    else:
        df["sample_weight"] = 1.0

    return df


def load_labeled_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)
    needed = [TEXT_COL, TARGET_COL, LABEL_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")
        df[c] = df[c].astype(str).str.strip()
    df = df[(df[TEXT_COL] != "") & (df[TARGET_COL] != "") & (df[LABEL_COL] != "")].copy()
    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)
    unknown = sorted(set(df[LABEL_COL]) - set(LABEL2ID.keys()))
    if unknown:
        raise ValueError(f"Unknown stance labels: {unknown}")
    df["label"] = df[LABEL_COL].map(LABEL2ID).astype(int)
    df = add_aux_labels(df)
    return df


def load_unlabeled_test_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)
    needed = [ID_COL, TEXT_COL, TARGET_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")
        df[c] = df[c].astype(str).str.strip()
    df = df[(df[ID_COL] != "") & (df[TEXT_COL] != "") & (df[TARGET_COL] != "")].copy()
    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)
    return df


train_df = load_labeled_dataframe(TRAIN_PATH)
dev_df = load_labeled_dataframe(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("\nTrain stance counts:", train_df[LABEL_COL].value_counts().to_dict())
print("Aux coverage -> sentiment:", (train_df['sent_label'] != IGNORE).sum(),
      "| sarcasm:", (train_df['sarc_label'] != IGNORE).sum(),
      "| explicitness:", (train_df['expl_label'] != IGNORE).sum())
print("Sample weight range: [{:.3f}, {:.3f}]".format(train_df['sample_weight'].min(), train_df['sample_weight'].max()))


Train shape: (3502, 19)
Dev shape: (619, 19)

Train stance counts: {'Favor': 2148, 'Against': 1021, 'None': 333}
Aux coverage -> sentiment: 3502 | sarcasm: 3502 | explicitness: 3110
Sample weight range: [0.677, 1.000]


In [8]:
# ============================================================
# 7) Dataset
# ============================================================
class StanceDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row[TARGET_COL],
            row[TEXT_COL],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if not self.is_test:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
            item["sent_label"] = torch.tensor(int(row["sent_label"]), dtype=torch.long)
            item["sarc_label"] = torch.tensor(int(row["sarc_label"]), dtype=torch.long)
            item["expl_label"] = torch.tensor(int(row["expl_label"]), dtype=torch.long)
            item["sample_weight"] = torch.tensor(float(row["sample_weight"]), dtype=torch.float)
        return item


In [9]:
# ============================================================
# 8) Multi-Task Model + Loss
# ============================================================
class SaliAIModel(nn.Module):
    def __init__(self, hf_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(hf_name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.stance_head = nn.Linear(h, 3)
        self.sent_head = nn.Linear(h, 3)
        self.sarc_head = nn.Linear(h, 2)
        self.expl_head = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out = self.encoder(**kwargs)
        pooled = self.dropout(out.last_hidden_state[:, 0])
        return {
            "stance": self.stance_head(pooled),
            "sent": self.sent_head(pooled),
            "sarc": self.sarc_head(pooled),
            "expl": self.expl_head(pooled),
        }


# Class weights from training distribution (inverse frequency)
counts = train_df["label"].value_counts().reindex([0, 1, 2]).fillna(0).values.astype(float)
class_weights = torch.tensor(len(train_df) / (3.0 * np.maximum(counts, 1.0)), dtype=torch.float).to(DEVICE)
print("Class weights (Against, Favor, None):", [round(w, 3) for w in class_weights.tolist()])


def safe_aux_ce(logits, labels):
    if (labels != IGNORE).any():
        return F.cross_entropy(logits, labels, ignore_index=IGNORE)
    return torch.tensor(0.0, device=logits.device)


def compute_loss(logits, batch):
    per_sample = F.cross_entropy(logits["stance"], batch["labels"], weight=class_weights, reduction="none")
    loss = (per_sample * batch["sample_weight"]).mean()
    loss = loss + AUX_W_SENT * safe_aux_ce(logits["sent"], batch["sent_label"])
    loss = loss + AUX_W_SARC * safe_aux_ce(logits["sarc"], batch["sarc_label"])
    loss = loss + AUX_W_EXPL * safe_aux_ce(logits["expl"], batch["expl_label"])
    return loss


Class weights (Against, Favor, None): [1.143, 0.543, 3.506]


In [10]:
# ============================================================
# 9) Tokenizer, Loaders, Optimizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_NAME)

train_ds = StanceDataset(train_df, tokenizer)
dev_ds = StanceDataset(dev_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

model = SaliAIModel(MODEL_HF_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_steps),
    num_training_steps=total_steps,
)


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# ============================================================
# 10) Metrics + Prediction Helpers
# ============================================================
ENC_KEYS = ("input_ids", "attention_mask", "token_type_ids")

def enc_batch(batch):
    return {k: batch[k].to(DEVICE) for k in ENC_KEYS if k in batch}


def compute_metrics(y_true, y_pred):
    f_against = f1_score(y_true, y_pred, labels=[0], average="macro", zero_division=0)
    f_favor = f1_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    f_none = f1_score(y_true, y_pred, labels=[2], average="macro", zero_division=0)
    return {
        "F_favor": f_favor,
        "F_against": f_against,
        "F_none": f_none,
        "Favg2": (f_favor + f_against) / 2.0,
        "Favg3": (f_favor + f_against + f_none) / 3.0,
        "Acc": accuracy_score(y_true, y_pred),
    }


@torch.no_grad()
def predict_labeled_loader(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        labels.extend(batch["labels"].numpy().tolist())
        logits = model(**enc_batch(batch))["stance"]
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
    return labels, preds


@torch.no_grad()
def predict_unlabeled_loader(model, loader):
    model.eval()
    preds = []
    for batch in tqdm(loader, desc="Predicting"):
        logits = model(**enc_batch(batch))["stance"]
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
    return preds


def build_per_target_results(eval_df, pred_ids, model_name):
    df = eval_df.copy()
    df["pred"] = pred_ids
    row = {"Model": model_name}
    for target in sorted(df[TARGET_COL].unique()):
        sub = df[df[TARGET_COL] == target]
        m = compute_metrics(sub["label"].tolist(), sub["pred"].tolist())
        key = target.replace(" ", "_")
        row[f"{key}_Favg2"] = round(m["Favg2"] * 100, 2)
        row[f"{key}_Favg3"] = round(m["Favg3"] * 100, 2)
    m = compute_metrics(df["label"].tolist(), df["pred"].tolist())
    row["F_favor"] = round(m["F_favor"] * 100, 2)
    row["F_against"] = round(m["F_against"] * 100, 2)
    row["F_none"] = round(m["F_none"] * 100, 2)
    row["Overall_Favg2"] = round(m["Favg2"] * 100, 2)
    row["Overall_Favg3"] = round(m["Favg3"] * 100, 2)
    row["Acc"] = round(m["Acc"] * 100, 2)
    return pd.DataFrame([row])


In [12]:
# ============================================================
# 11) Training + Dev Evaluation (keeps best Favg2 checkpoint)
# ============================================================
model_base_dir = f"{OUTPUT_DIR}/{MODEL_DISPLAY_NAME}"
os.makedirs(model_base_dir, exist_ok=True)
BEST_CKPT = f"{model_base_dir}/best_favg2.pt"

history = []
best_dev_favg2 = -1.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"{MODEL_DISPLAY_NAME} | Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        enc = enc_batch(batch)
        tgt = {k: batch[k].to(DEVICE) for k in ("labels", "sent_label", "sarc_label", "expl_label", "sample_weight")}
        optimizer.zero_grad()
        logits = model(**enc)
        loss = compute_loss(logits, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = total_loss / max(len(train_loader), 1)
    dev_labels, dev_preds = predict_labeled_loader(model, dev_loader)
    m = compute_metrics(dev_labels, dev_preds)

    history.append({"epoch": epoch + 1, "train_loss": train_loss,
                    "dev_Favg2": m["Favg2"] * 100, "dev_Favg3": m["Favg3"] * 100, "dev_Acc": m["Acc"] * 100})
    print(f"Epoch {epoch+1:02d} | train_loss={train_loss:.4f} | dev_Favg2={m['Favg2']*100:.2f} | dev_Favg3={m['Favg3']*100:.2f} | dev_acc={m['Acc']*100:.2f}")

    if m["Favg2"] > best_dev_favg2:
        best_dev_favg2 = m["Favg2"]
        torch.save(model.state_dict(), BEST_CKPT)
        tokenizer.save_pretrained(model_base_dir)
        print(f"  -> New best Favg2 = {best_dev_favg2*100:.2f} (checkpoint saved)")

pd.DataFrame(history).to_csv(f"{model_base_dir}/training_history.csv", index=False)
print("\nBest dev Favg2:", round(best_dev_favg2 * 100, 2))


SaliAI-MARBERTv2-MTL | Epoch 1/10:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 01 | train_loss=1.3827 | dev_Favg2=69.86 | dev_Favg3=55.66 | dev_acc=63.33
  -> New best Favg2 = 69.86 (checkpoint saved)


SaliAI-MARBERTv2-MTL | Epoch 2/10:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 02 | train_loss=0.9859 | dev_Favg2=78.30 | dev_Favg3=66.92 | dev_acc=75.77
  -> New best Favg2 = 78.30 (checkpoint saved)


SaliAI-MARBERTv2-MTL | Epoch 3/10:   0%|          | 0/110 [00:00<?, ?it/s]

...

Epoch 08 | train_loss=0.2570 | dev_Favg2=84.18 | dev_Favg3=70.22 | dev_acc=82.23
  -> New best Favg2 = 84.18 (checkpoint saved)


SaliAI-MARBERTv2-MTL | Epoch 9/10:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 09 | train_loss=0.2436 | dev_Favg2=84.16 | dev_Favg3=70.27 | dev_acc=81.74


SaliAI-MARBERTv2-MTL | Epoch 10/10:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 10 | train_loss=0.2231 | dev_Favg2=85.03 | dev_Favg3=70.41 | dev_acc=82.55
  -> New best Favg2 = 85.03 (checkpoint saved)

Best dev Favg2: 85.03


In [13]:
# ============================================================
# 12) Final Dev Evaluation (best checkpoint)
# ============================================================
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))

dev_labels, dev_preds = predict_labeled_loader(model, dev_loader)
m = compute_metrics(dev_labels, dev_preds)

print("FINAL DEV RESULTS USING BEST_FAVG2 CHECKPOINT")
print(f"F_favor:   {m['F_favor']*100:.2f}")
print(f"F_against: {m['F_against']*100:.2f}")
print(f"F_none:    {m['F_none']*100:.2f}")
print(f"Favg2:     {m['Favg2']*100:.2f}")
print(f"Favg3:     {m['Favg3']*100:.2f}")
print(f"Accuracy:  {m['Acc']*100:.2f}")

print("\nDEV CLASSIFICATION REPORT")
print(classification_report(dev_labels, dev_preds, target_names=["Against", "Favor", "None"], digits=4, zero_division=0))

per_target = build_per_target_results(dev_df, dev_preds, MODEL_DISPLAY_NAME)
print("\nDEV RESULTS PER TARGET")
display(per_target)

dev_out = dev_df[[TEXT_COL, TARGET_COL, LABEL_COL]].copy()
dev_out["pred"] = [ID2LABEL[p] for p in dev_preds]
dev_out.to_csv(f"{model_base_dir}/dev_predictions.csv", index=False)
per_target.to_csv(f"{model_base_dir}/dev_results_per_target.csv", index=False)
print("Saved predictions and per-target results to:", model_base_dir)


FINAL DEV RESULTS USING BEST_FAVG2 CHECKPOINT
F_favor:   88.68
F_against: 81.38
F_none:    41.18
Favg2:     85.03
Favg3:     70.41
Accuracy:  82.55

DEV CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Against     0.7806    0.8500    0.8138       180
       Favor     0.8868    0.8868    0.8868       380
        None     0.4884    0.3559    0.4118        59

    accuracy                         0.8255       619
   macro avg     0.7186    0.6976    0.7041       619
weighted avg     0.8180    0.8255    0.8203       619


DEV RESULTS PER TARGET


,Model,Covid_Vaccine_Favg2,Covid_Vaccine_Favg3,Digital_Transformation_Favg2,Digital_Transformation_Favg3,Women_empowerment_Favg2,Women_empowerment_Favg3,F_favor,F_against,F_none,Overall_Favg2,Overall_Favg3,Acc
0,SaliAI-MARBERTv2-MTL,81.98,64.8,80.21,71.99,87.87,71.91,88.68,81.38,41.18,85.03,70.41,82.55


Saved predictions and per-target results to: ./saliai_outputs/SaliAI-MARBERTv2-MTL


In [ ]:
# ============================================================
# 13) Submission Generation (run on July 20 when test files arrive)
# ============================================================
def generate_submission(test_path, output_filename):
    if not os.path.exists(test_path):
        print(f"[SKIPPED] {test_path} not found yet - blind test is released on July 20. Re-run this cell then.")
        return None
    test_df = load_unlabeled_test_dataframe(test_path)
    test_ds = StanceDataset(test_df, tokenizer, is_test=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    pred_ids = predict_unlabeled_loader(model, test_loader)
    submission = pd.DataFrame({ID_COL: test_df[ID_COL], LABEL_COL: [ID2LABEL[p] for p in pred_ids]})
    out_path = f"{model_base_dir}/{output_filename}"
    submission.to_csv(out_path, index=False)
    print("Submission saved to:", out_path)
    return submission

submission_seen = generate_submission(TEST_SEEN_PATH, "submission_seen.csv")
submission_unseen = generate_submission(TEST_UNSEEN_PATH, "submission_unseen.csv")


## Notes for the system description paper

Keep track of these for the write-up:

- Baseline (AraBERTv0.2-Twitter): Favg2 = **83.67** on dev
- This model: record your Favg2 here after each run
- Ablations worth reporting: with/without confidence weighting, with/without each auxiliary head
- All auxiliary supervision comes from columns already in Mawqif-v2 (`sentiment`, `sarcasm`, `stance:confidence`, `favor_reason`/`against_reason` explicitness) - no external data used
